<a href="https://colab.research.google.com/github/Varapurnakarthik1/ELECTRICAL-NETWORK-NATURAL-MODES-/blob/main/Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# FRONTEND - IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("Frontend loaded successfully!")

Frontend loaded successfully!


In [2]:

# ============================================================
# FRONTEND - INPUT PANEL
# ============================================================

title = widgets.HTML("""
<h1 style="text-align:center;">
    ⚡ Electrical Network Natural Modes
</h1>

<p style="text-align:center; font-size:16px;">
    B.Tech Mathematics Mini Project — B9
</p>

<hr>
""")

# ---------- Inductance inputs ----------

L1_box = widgets.FloatText(
    value=10,
    description="L1 (mH):",
    style={'description_width': 'initial'}
)

L2_box = widgets.FloatText(
    value=10,
    description="L2 (mH):",
    style={'description_width': 'initial'}
)

L3_box = widgets.FloatText(
    value=10,
    description="L3 (mH):",
    style={'description_width': 'initial'}
)

# ---------- Mutual inductance ----------

M12_box = widgets.FloatText(
    value=2,
    description="M12 (mH):",
    style={'description_width': 'initial'}
)

M13_box = widgets.FloatText(
    value=2,
    description="M13 (mH):",
    style={'description_width': 'initial'}
)

M23_box = widgets.FloatText(
    value=2,
    description="M23 (mH):",
    style={'description_width': 'initial'}
)

# ---------- Capacitance ----------

C1_box = widgets.FloatText(
    value=100,
    description="C1 (nF):",
    style={'description_width': 'initial'}
)

C2_box = widgets.FloatText(
    value=100,
    description="C2 (nF):",
    style={'description_width': 'initial'}
)

C3_box = widgets.FloatText(
    value=100,
    description="C3 (nF):",
    style={'description_width': 'initial'}
)

# ---------- Resistance ----------

R_box = widgets.FloatText(
    value=1,
    description="R (Ω):",
    style={'description_width': 'initial'}
)

# ---------- Calculate button ----------

calculate_button = widgets.Button(
    description="Calculate Natural Modes",
    button_style="success",
    icon="calculator"
)

output_area = widgets.Output()

display(
    title,

    widgets.HTML("<h3>Inductances</h3>"),
    widgets.HBox([L1_box, L2_box, L3_box]),

    widgets.HTML("<h3>Mutual Inductances</h3>"),
    widgets.HBox([M12_box, M13_box, M23_box]),

    widgets.HTML("<h3>Capacitances</h3>"),
    widgets.HBox([C1_box, C2_box, C3_box]),

    widgets.HTML("<h3>Resonance Parameters</h3>"),
    R_box,

    calculate_button,
    output_area
)

HTML(value='\n<h1 style="text-align:center;">\n    ⚡ Electrical Network Natural Modes\n</h1>\n\n<p style="text…

HTML(value='<h3>Inductances</h3>')

HTML(value='<h3>Mutual Inductances</h3>')

HTML(value='<h3>Capacitances</h3>')

HTML(value='<h3>Resonance Parameters</h3>')

FloatText(value=1.0, description='R (Ω):', style=DescriptionStyle(description_width='initial'))

Button(button_style='success', description='Calculate Natural Modes', icon='calculator', style=ButtonStyle())

Output()

In [3]:

# ============================================================
# FRONTEND → MATHEMATICAL ENGINE
# ============================================================

def calculate_project():

    # Convert units
    L1 = L1_box.value * 1e-3
    L2 = L2_box.value * 1e-3
    L3 = L3_box.value * 1e-3

    M12 = M12_box.value * 1e-3
    M13 = M13_box.value * 1e-3
    M23 = M23_box.value * 1e-3

    C1 = C1_box.value * 1e-9
    C2 = C2_box.value * 1e-9
    C3 = C3_box.value * 1e-9

    R = R_box.value

    # --------------------------------------------------------
    # Inductance matrix
    # --------------------------------------------------------

    L = np.array([
        [L1, M12, M13],
        [M12, L2, M23],
        [M13, M23, L3]
    ])

    # --------------------------------------------------------
    # Capacitance matrix
    # --------------------------------------------------------

    C = np.diag([C1, C2, C3])

    C_inv = np.linalg.inv(C)

    # --------------------------------------------------------
    # Eigenvalue matrix
    # A u = lambda u
    # lambda = omega^2
    # --------------------------------------------------------

    A = np.linalg.solve(L, C_inv)

    eigenvalues, eigenvectors = np.linalg.eig(A)

    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)

    omega = np.sqrt(eigenvalues)
    frequency = omega / (2*np.pi)

    # Sort
    order = np.argsort(frequency)

    eigenvalues = eigenvalues[order]
    omega = omega[order]
    frequency = frequency[order]
    eigenvectors = eigenvectors[:, order]

    # Normalize eigenvectors
    for k in range(3):
        eigenvectors[:, k] /= np.max(
            np.abs(eigenvectors[:, k])
        )

    return (
        L, C, A,
        eigenvalues,
        omega,
        frequency,
        eigenvectors,
        R
    )

In [4]:

# ============================================================
# CALCULATE BUTTON ACTION
# ============================================================

def on_calculate_clicked(button):

    with output_area:

        clear_output()

        try:

            (
                L,
                C,
                A,
                eigenvalues,
                omega,
                frequency,
                eigenvectors,
                R
            ) = calculate_project()

            # Store globally for other visualizations
            globals()["L_matrix_front"] = L
            globals()["C_matrix_front"] = C
            globals()["A_front"] = A
            globals()["eigenvalues_front"] = eigenvalues
            globals()["omega_front"] = omega
            globals()["frequency_front"] = frequency
            globals()["eigenvectors_front"] = eigenvectors

            # ------------------------------------------------
            # Header
            # ------------------------------------------------

            display(HTML("""
            <hr>
            <h2>📊 Natural Mode Analysis</h2>
            """))

            # ------------------------------------------------
            # Frequency table
            # ------------------------------------------------

            html = """
            <table style="
                border-collapse:collapse;
                width:100%;
                text-align:center;
                font-size:15px;
            ">

            <tr style="background:#222;color:white;">
                <th>Mode</th>
                <th>Eigenvalue ω²</th>
                <th>ω (rad/s)</th>
                <th>Natural Frequency (Hz)</th>
            </tr>
            """

            for k in range(3):

                html += f"""
                <tr>
                    <td>Mode {k+1}</td>
                    <td>{eigenvalues[k]:.4e}</td>
                    <td>{omega[k]:.3f}</td>
                    <td><b>{frequency[k]:.3f}</b></td>
                </tr>
                """

            html += "</table>"

            display(HTML(html))

            # ------------------------------------------------
            # Eigenvectors
            # ------------------------------------------------

            display(HTML("<h3>🔢 Current Distribution</h3>"))

            html = """
            <table style="
                border-collapse:collapse;
                width:100%;
                text-align:center;
            ">

            <tr style="background:#222;color:white;">
                <th>Circuit</th>
                <th>Mode 1</th>
                <th>Mode 2</th>
                <th>Mode 3</th>
            </tr>
            """

            for i in range(3):

                html += f"""
                <tr>
                    <td><b>i{i+1}</b></td>
                    <td>{eigenvectors[i,0]:.4f}</td>
                    <td>{eigenvectors[i,1]:.4f}</td>
                    <td>{eigenvectors[i,2]:.4f}</td>
                </tr>
                """

            html += "</table>"

            display(HTML(html))

            # ------------------------------------------------
            # Mathematical summary
            # ------------------------------------------------

            display(HTML(f"""
            <br>
            <div style="
                padding:15px;
                border-radius:10px;
                background:#f2f2f2;
            ">
            <b>Mathematical Model:</b><br><br>

            L d²i/dt² + C⁻¹i = 0
            <br><br>

            L⁻¹C⁻¹u = ω²u
            <br><br>

            <b>Eigenvalue = ω²</b><br>
            <b>Eigenvector = Current Distribution</b>
            </div>
            """))

        except Exception as e:

            display(HTML(
                f"""
                <h3 style="color:red;">
                ❌ Error
                </h3>
                <p>{e}</p>
                """
            ))


calculate_button.on_click(on_calculate_clicked)

In [5]:

# ============================================================
# VISUALIZATION DASHBOARD
# ============================================================

def show_charts():

    if "frequency_front" not in globals():
        print("First click 'Calculate Natural Modes'.")
        return

    frequency = frequency_front
    eigenvectors = eigenvectors_front

    # --------------------------------------------------------
    # Natural frequencies
    # --------------------------------------------------------

    plt.figure(figsize=(9,5))

    plt.stem(
        [1,2,3],
        frequency
    )

    plt.xlabel("Natural Mode")
    plt.ylabel("Frequency (Hz)")
    plt.title("Natural Frequencies")

    plt.xticks(
        [1,2,3],
        ["Mode 1","Mode 2","Mode 3"]
    )

    plt.grid(axis="y", alpha=0.3)

    plt.show()


    # --------------------------------------------------------
    # Current distribution
    # --------------------------------------------------------

    x = np.arange(3)
    width = 0.25

    plt.figure(figsize=(10,5))

    for k in range(3):

        plt.bar(
            x + (k-1)*width,
            eigenvectors[:,k],
            width,
            label=f"Mode {k+1}"
        )

    plt.axhline(0, linewidth=1)

    plt.xticks(
        x,
        ["i1","i2","i3"]
    )

    plt.xlabel("Circuit Current")
    plt.ylabel("Normalized Current")
    plt.title("Eigenvector / Current Distribution")

    plt.legend()
    plt.grid(axis="y", alpha=0.3)

    plt.show()

In [6]:

# ============================================================
# CHART BUTTON
# ============================================================

chart_button = widgets.Button(
    description="Show Mathematical Charts",
    button_style="info",
    icon="bar-chart"
)

chart_output = widgets.Output()


def on_chart_clicked(button):

    with chart_output:

        clear_output()

        show_charts()


chart_button.on_click(on_chart_clicked)

display(chart_button)
display(chart_output)

Button(button_style='info', description='Show Mathematical Charts', icon='bar-chart', style=ButtonStyle())

Output()